In [1]:
%pip install requests numpy pandas
%pip install scikit-learn
import os
import pandas as pd
from sklearn.datasets import load_breast_cancer
import numpy as np

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Load Databricks token from file
with open('databricks_token.txt', 'r') as f: #generar archivo txt y pegar el token de databricks
    token = f.read().strip()
os.environ['DATABRICKS_TOKEN'] = token

In [3]:
# Load new transactions
new_transaction_pd = pd.DataFrame([{
    'step':             42.0,
    'amount':           9690.61,
    'oldbalanceOrg':    52005.0,
    'newbalanceOrig':   42314.39,
    'oldbalanceDest':   55317.03,
    'newbalanceDest':   65007.64,
    'isFlaggedFraud':   0.0,
    'errorBalanceOrig': 0.0,
    'errorBalanceDest': 0.0,
    'origZeroBalance':  0.0,
    'destZeroBalance':  0.0,
    'type_idx':         0.0
}])

new_transaction_fraud_pd = pd.DataFrame([{
    'step':             45.0,         # índice 0
    'amount':           4919642.57,   # índice 1
    'oldbalanceOrg':    4919642.57,   # índice 2
    'newbalanceOrig':   0.0,          # índice 3 → ausente = 0
    'oldbalanceDest':   1414095.92,   # índice 4
    'newbalanceDest':   6333738.49,   # índice 5
    'isFlaggedFraud':   0.0,          # índice 6 → ausente = 0
    'errorBalanceOrig': 0.0,          # índice 7 → ausente = 0
    'errorBalanceDest': 0.0,          # índice 8 → ausente = 0
    'origZeroBalance':  0.0,          # índice 9 → ausente = 0
    'destZeroBalance':  0.0,          # índice 10 → ausente = 0
    'type_idx':         0.0           # índice 11 → ausente = 0
}])

In [4]:
new_transaction_pd

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFlaggedFraud,errorBalanceOrig,errorBalanceDest,origZeroBalance,destZeroBalance,type_idx
0,42.0,9690.61,52005.0,42314.39,55317.03,65007.64,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
new_transaction_fraud_pd

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFlaggedFraud,errorBalanceOrig,errorBalanceDest,origZeroBalance,destZeroBalance,type_idx
0,45.0,4919642.57,4919642.57,0.0,1414095.92,6333738.49,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:

import os
import requests
import numpy as np
import pandas as pd
import json

def create_tf_serving_json(data):
    return {'inputs': {name: data[name].tolist() for name in data.keys()} if isinstance(data, dict) else data.tolist()}

def score_model(dataset):
    url = 'https://dbc-256f75fe-2a85.cloud.databricks.com/serving-endpoints/RF_Fraun_sklearn/invocations' #cambiar endpont del modelo
    headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}
    ds_dict = {'dataframe_split': dataset.to_dict(orient='split')} if isinstance(dataset, pd.DataFrame) else create_tf_serving_json(dataset)
    data_json = json.dumps(ds_dict, allow_nan=True)
    response = requests.request(method='POST', headers=headers, url=url, data=data_json)
    if response.status_code != 200:
        raise Exception(f'Request failed with status {response.status_code}, {response.text}')
    return response.json()

In [8]:
result = score_model(new_transaction_pd)

prediction = result["predictions"][0]
status = "⚠️ FRAUDE DETECTADO" if prediction == 1.0 else "✅ TRANSACCIÓN NORMAL"
print(f"Veredicto: {status}")
print(f"Raw response: {result}")

Veredicto: ✅ TRANSACCIÓN NORMAL
Raw response: {'predictions': [0]}


In [9]:
result = score_model(new_transaction_fraud_pd)

prediction = result["predictions"][0]
status = "⚠️ FRAUDE DETECTADO" if prediction == 1.0 else "✅ TRANSACCIÓN NORMAL"
print(f"Veredicto: {status}")
print(f"Raw response: {result}")

Veredicto: ⚠️ FRAUDE DETECTADO
Raw response: {'predictions': [1]}
